In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression,ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,root_mean_squared_error,confusion_matrix,roc_auc_score,log_loss
from sklearn.metrics import f1_score,accuracy_score,recall_score,precision_score,classification_report
from tqdm import tqdm
import os
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler,PolynomialFeatures
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
import datetime as dt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis,QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import BernoulliNB,GaussianNB
from sklearn.svm import SVC

In [2]:
hr = pd.read_csv(r"../Datasets/HR_comma_sep.csv")
hr

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [3]:
le=LabelEncoder() 
hr['Department']=le.fit_transform(hr['Department'])
hr['salary']=le.fit_transform(hr['salary'])

X = hr.drop('left',axis=1)
print(X)
y=hr['left']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26, stratify=hr['left'])

       satisfaction_level  last_evaluation  number_project  \
0                    0.38             0.53               2   
1                    0.80             0.86               5   
2                    0.10             0.77               6   
3                    0.92             0.85               5   
4                    0.89             1.00               5   
...                   ...              ...             ...   
14990                0.40             0.57               2   
14991                0.37             0.48               2   
14992                0.37             0.53               2   
14993                0.11             0.96               6   
14994                0.37             0.52               2   

       average_montly_hours  time_spend_company  Work_accident  \
0                       157                   3              0   
1                       262                   6              0   
2                       247                   4          

In [5]:
Cs = np.linspace(0.001, 1, 10)
scores = []
for c in Cs:
    svm = SVC(kernel='linear', C=c)
    svm.fit(X_train, y_train)
    y_pred = svm.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    scores.append((c,accuracy))
print("Accuracy:", accuracy)
df_scores = pd.DataFrame(scores, columns=['c','accuracy'])
df_scores.sort_values( 'accuracy', ascending=False).head()

Accuracy: 0.7763947543898644


,c,accuracy
3,0.334,0.799066
4,0.445,0.793065
5,0.556,0.780840
6,0.667,0.780396
7,0.778,0.777951


### svm with log loss

In [6]:
Cs = np.linspace(0.001, 1, 20)
scores = []
for c in Cs:
    svm = SVC(kernel='linear', C=c, probability=True,random_state=26)
    svm.fit(X_train, y_train)
    y_pred_prob = svm.predict_proba(X_test)
    accuracy = log_loss(y_test, y_pred_prob)
    scores.append([c,accuracy])
df_scores = pd.DataFrame(scores, columns=['c','log_loss'])
df_scores.sort_values( 'log_loss', ascending=True).head()

### Polynomial Kernel: Tested curved boundaries with different degrees (2, 3, and 4).

In [ ]:
Cs = np.linspace(0.001, 5, 20)
Ds=[2,3,4]
scores = []
for c in Cs:
    for d in Ds:
        svm = SVC(kernel='poly', C=c, probability=True,random_state=26,degree=d)
        svm.fit(X_train, y_train)
        y_pred_prob = svm.predict_proba(X_test)
        scores.append([c,d,log_loss(y_test, y_pred_prob)])
df_scores = pd.DataFrame(scores, columns=['c','d','log_loss'])
df_scores.sort_values( 'log_loss', ascending=True).head()

### RBF (Radial Basis Function) Kernel: Tested a highly flexible boundary that uses a "distance" logic (Gamma).

In [ ]:
Cs = np.linspace(0.001, 5, 20)
Gs = np.linspace(0.001,5,20)

scores = []
for c in Cs:
    for g in Gs:
        svm = SVC(kernel='rbf', C=c, probability=True,random_state=26,gamma = g)
        svm.fit(X_train, y_train)
        y_pred_prob = svm.predict_proba(X_test)
        scores.append([c,g,log_loss(y_test, y_pred_prob)])

df_scores = pd.DataFrame(scores, columns=['c',g,'log_loss'])
df_scores.sort_values( 'log_loss', ascending=True).head()

### sigmoid

In [ ]:
Cs = np.linspace(0.001, 5, 20)
coefs = np.linspace(-5, 5, 20)

scores = []
for c in Cs:
    for cf in coefs: 
        svm = SVC(kernel='sigmoid', C=c, probability=True, random_state=26, coef0=cf)
        svm.fit(X_train, y_train)
        y_pred_prob = svm.predict_proba(X_test)
        scores.append([c, cf, log_loss(y_test, y_pred_prob)])

df_scores = pd.DataFrame(scores, columns=['c', 'coef0', 'log_loss'])

df_scores.sort_values('log_loss', ascending=True).head()

### OneHotEncoder

In [ ]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")
trans = ColumnTransformer(
    transformers=[("OHE", ohe,make_column_selector(dtype_include=object))],remainder="passthrough",
    verbose_feature_names_out=False).set_output(transform="pandas")

X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)

In [ ]:
from tqdm import tqdm

Cs = np.linspace(0.001, 1, 20)
scores = []
for c in tqdm(Cs):
    svm = SVC(kernel='linear', C=c, probability=True,random_state=26)
    svm.fit(X_trn_scl, y_train)
    y_pred_prob = svm.predict_proba(X_tst_scl)
    accuracy = log_loss(y_test, y_pred_prob)
    scores.append([c,accuracy])

df_scores = pd.DataFrame(scores, columns=['c','log_loss'])
df_scores.sort_values( 'log_loss', ascending=True).head()

### RBF (Radial Basis Function) Kernel: Tested a highly flexible boundary that uses a "distance" logic (Gamma).

In [ ]:
Cs = np.linspace(0.001, 5, 20)
Gs = np.linspace(0.001,5,20)

scores = []
for c in Cs:
    for g in Gs:
        svm = SVC(kernel='rbf', C=c, probability=True,random_state=26,gamma = g)
         svm.fit(X_trn_scl, y_train)
        y_pred_prob = svm.predict_proba(X_tst_scl)
        log_loss(y_test, y_pred_prob)
        scores.append([c,g,log_loss(y_test, y_pred_prob))

df_scores = pd.DataFrame(scores, columns=['c','log_loss'])
df_scores.sort_values( 'log_loss', ascending=True).head()

In [ ]:
from sklearn.compose import make_column_selector
from sklearn.compose import  ColumnTransformer

ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")
trans = ColumnTransformer(
    transformers=[("OHE", ohe,make_column_selector(dtype_include=object))],remainder="passthrough",
    verbose_feature_names_out=False).set_output(transform="pandas")

X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)

In [ ]:
scalar = StandardScaler()
X_trn_scl = scalar.fit_transform(X_trn_ohe)
X_tst_scl = scalar.transform(X_tst_ohe)

In [ ]:
from tqdm import tqdm

Cs = np.linspace(0.001, 1, 20)
scores = []
for c in tqdm(Cs):
    svm = SVC(kernel='linear', C=c, probability=True,random_state=26)
    svm.fit(X_trn_scl, y_train)
    y_pred_prob = svm.predict_proba(X_tst_scl)
    accuracy = log_loss(y_test, y_pred_prob)
    scores.append([c,accuracy])

df_scores = pd.DataFrame(scores, columns=['c','log_loss'])
df_scores.sort_values( 'log_loss', ascending=True).head()

In [ ]:
Cs = np.linspace(0.001, 5, 20)
Gs = np.linspace(0.001,5,20)

scores = []
for c in Cs:
    for g in Gs:
        svm = SVC(kernel='rbf', C=c, probability=True,random_state=26,gamma = g)
        svm.fit(X_trn_scl, y_train)
        y_pred_prob = svm.predict_proba(X_tst_scl)
        log_loss(y_test, y_pred_prob)
        scores.append([c,g,log_loss(y_test, y_pred_prob)])

df_scores = pd.DataFrame(scores, columns=['c','g','log_loss'])
df_scores.sort_values( 'log_loss', ascending=True).head()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.inspection import DecisionBoundaryDisplay

# 1. Pick 2 features to visualize (satisfaction and evaluation)
# We use .iloc[:, [0, 1]] to get the first two columns easily
X_plot = X.iloc[:, [0, 1]] 
y_plot = y

# 2. Create and train a simple Linear SVM
model = SVC(kernel='linear', C=1.0)
model.fit(X_plot, y_plot)

# 3. Draw the decision boundary (the "colored background")
DecisionBoundaryDisplay.from_estimator(
    model, 
    X_plot, 
    response_method="predict",
    cmap=plt.cm.coolwarm, 
    alpha=0.8
)

# 4. Drop the actual data points on top
plt.scatter(X_plot.iloc[:, 0], X_plot.iloc[:, 1], c=y_plot, cmap=plt.cm.coolwarm, edgecolors='k')

# 5. Add labels
plt.title("Linear SVM Decision Boundary")
plt.xlabel("Satisfaction Level")
plt.ylabel("Last Evaluation")
plt.show()